# RewardMaxxing v2 — reproducible Colab execution

Choose **A100 for the first pilot**, then compare L4 cost per completed update. Check actual CU/hour in Colab's Resources panel. Run cells individually: the sweep starts paid GPU work.

Push the reviewed `refactor/v2` branch first. This notebook clones it and pins the session to a commit. See `docs/EXECUTION.md` for Git, budgets, pilot thresholds and low-memory evaluation. The default STIM metric remains a retokenized partial-corpus proxy.


In [ ]:
from google.colab import drive
from pathlib import Path
import subprocess, sys, json, os

drive.mount('/content/drive')
ROOT = Path('/content/rewardmaxxing-v2')
REPO = 'https://github.com/MuhammadBinTariq/STIM4RM.git'
BRANCH = 'refactor/v2'
COMMIT = ''  # Paste the full SHA from git rev-parse HEAD before a formal experiment.
if not ROOT.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO, str(ROOT)], check=True)
else:
    assert (ROOT / '.git').is_dir(), 'Use a fresh path; this directory is not a Git checkout'
    assert not subprocess.check_output(['git','-C',str(ROOT),'status','--porcelain'], text=True).strip(), 'Save local edits before updating'
    assert subprocess.check_output(['git','-C',str(ROOT),'remote','get-url','origin'], text=True).strip() == REPO
    subprocess.run(['git','-C',str(ROOT),'fetch','origin',BRANCH,'--tags'], check=True)
target = COMMIT or 'origin/' + BRANCH
subprocess.run(['git','-C',str(ROOT),'checkout','--detach',target], check=True)
SHA = subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'], text=True).strip()
print('Pinned source:', SHA)
OUTPUT = Path('/content/drive/MyDrive/RewardMaxxing-v2')
OUTPUT.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = '/content/huggingface'  # Local SSD, not live Drive model caches.


Install the tested dependency versions in a separate virtual environment. The CUDA build requires a compatible Colab driver; installation or GPU failures stop execution. This may download several GB on the first session.


In [ ]:
VENV = Path('/content/rewardmaxxing-venv')
if not (VENV / 'bin/python').exists():
    subprocess.run([sys.executable, '-m', 'venv', str(VENV)], check=True)
PY = str(VENV / 'bin/python')
subprocess.run([PY, '-m', 'pip', 'install', 'torch==2.6.0', '--index-url', 'https://download.pytorch.org/whl/cu124'], check=True)
subprocess.run([PY, '-m', 'pip', 'install', '-r', str(ROOT / 'requirements-v2.txt')], check=True)
subprocess.run([PY, 'tests/smoke_test.py'], cwd=ROOT, check=True)


Resolve model, tokenizer, PRM, and dataset commits **once**. Both arms reuse this file. The provided configuration explicitly acknowledges a retokenized, stage-one-corpus proxy. Do not replace it with a native-token claim.


In [ ]:
RESOLVED = OUTPUT / 'resolved.json'
if not RESOLVED.exists():
    subprocess.run([PY, '-m', 'src.prepare', '--config', 'configs/rewardmaxxing.json', '--output', str(RESOLVED)], cwd=ROOT, check=True)
print(RESOLVED.read_text())


## Pilot: matched 25-update runs

2 prompts × 4 rollouts = 8 responses/update. Generation microbatch is 2, update microbatch is 1. Baseline STIM monitoring is enabled with lambda=0. Run the same measured recipe on each GPU before choosing by CU/update. Use separate roots for hardware pilots; do not mix hardware within a paired seed.


In [ ]:
PILOT = OUTPUT / 'pilot-a100-lambda01'
PRIMARY = OUTPUT / 'primary-lambda01'

def execute(arm, phase, seed=42, root=PRIMARY, extra=()):
    command = [PY, 'scripts/colab_run.py', '--resolved-config', str(RESOLVED),
               '--output-root', str(root), '--arm', arm, '--phase', phase, '--seed', str(seed),
               '--stim-lambda', '0.1', '--monitor-stim', *map(str, extra)]
    subprocess.run(command, cwd=ROOT, check=True)

for arm in ['baseline', 'rewardmaxxing']:
    execute(arm, 'pilot', root=PILOT, extra=['--pilot-steps', '25'])


Inspect pilots before scheduling all seeds. Cold API latency and boundary rejection rates determine whether this configuration is viable. No automatic lambda/gating changes are made when coverage fails.


In [ ]:
for arm in ['baseline', 'rewardmaxxing']:
    run = PILOT / f'{arm}-seed42'
    print(arm, (run / 'pilot-summary.json').read_text())
    print('Storage:', (run / 'storage.json').read_text())
    subprocess.run([PY, 'scripts/export_metrics.py', '--run-dir', str(run), '--output', str(run/'metrics.csv')], cwd=ROOT, check=True)

# These values come from YOUR current Resources panel, not a hard-coded GPU tariff.
CU_PER_HOUR = None
if CU_PER_HOUR is not None:
    seconds = []
    for arm in ['baseline', 'rewardmaxxing']:
        files = sorted((PILOT/f'{arm}-seed42/metrics').glob('step-*.json'))[5:]
        seconds.append(sum(json.loads(p.read_text())['elapsed_seconds'] for p in files)/len(files))
    hours = 3 * 256 * sum(seconds) / 3600
    print('Estimated training only:', hours, 'hours;', hours * CU_PER_HOUR, 'CU')
    print('Add measured evaluation, initialization and retry costs; keep a 160-CU reserve.')


## Primary sweep

Freeze the recipe using development data first. This starts six sequential runs, each 256 updates (512 training prompts, one epoch), three paired seeds. For a development frontier, instead stop at cumulative 64,128,256 updates and evaluate each before retention deletes older adapters. The CLI flag is `--train-steps`.


In [ ]:
assert COMMIT and SHA == COMMIT, 'Set COMMIT to the full tested source SHA before the formal sweep'
# Starts paid GPU work; execute only after reviewing the pilot and CU estimate.
for seed in [42, 43, 44]:
    for arm in ['baseline', 'rewardmaxxing']:
        execute(arm, 'train', seed, extra=['--train-steps','256'])


## Pinned twins and two-process evaluation

The following cell prepares one fixed instance per original question from GSM-Symbolic main (numeric perturbations) and p1 (additional structure). These are final test data. Do not use their scores to choose lambda. `--paired-only` reports GSM8K accuracy on the corresponding original-question panel; run a separate full-test evaluation for headline GSM8K accuracy.

Generation + exact KL + STIM run with **no PRM loaded**. A second process then loads only VersaPRM and reuses the exact saved responses. This removes simultaneous policy/PRM residency during evaluation.


In [ ]:
TWINS_DIR = OUTPUT / 'evaluation-data/symbolic-one-instance'
if not (TWINS_DIR / 'manifest.json').exists():
    subprocess.run([PY,'scripts/prepare_twins.py','--output-dir',str(TWINS_DIR),
                    '--subsets','main','p1','--instances','1'], cwd=ROOT, check=True)
TWINS = [TWINS_DIR/'main.jsonl', TWINS_DIR/'p1.jsonl']
for seed in [42,43,44]:
    for arm in ['baseline','rewardmaxxing']:
        run = PRIMARY / f'{arm}-seed{seed}'
        generated, scored = run/'paired-generations.json', run/'paired-scored.json'
        if not generated.exists():
            execute(arm, 'evaluate', seed, extra=['--skip-prm','--paired-only',
                '--eval-output',generated.name,'--twins',*TWINS])
        if not scored.exists():
            subprocess.run([PY,'-m','src.evaluate','score-prm','--input',str(generated),
                '--output',str(scored),'--config',str(run/'config.json')], cwd=ROOT, check=True)


In [ ]:
baseline = [str(PRIMARY / f'baseline-seed{s}/paired-scored.json') for s in [42,43,44]]
treatment = [str(PRIMARY / f'rewardmaxxing-seed{s}/paired-scored.json') for s in [42,43,44]]
subprocess.run([PY, '-m', 'src.evaluate', 'compare', '--baseline', *baseline,
                '--rewardmaxxing', *treatment, '--output', str(PRIMARY / 'paired-comparison.json')], cwd=ROOT, check=True)
print((PRIMARY / 'paired-comparison.json').read_text())


## Inspect local learning curves
Run after exporting pilot CSVs. This uses the notebook plotting environment, not an external tracking service.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
keys = ['task_reward','correct','kl','stim_rho','stim_valid_fraction','cache_hit_rate',
        'stim_wait_seconds','cuda_allocated_gb','cuda_reserved_gb']
fig, axes = plt.subplots(3, 3, figsize=(15, 10))
for arm in ['baseline','rewardmaxxing']:
    frame = pd.read_csv(PILOT/f'{arm}-seed42/metrics.csv')
    for ax, key in zip(axes.flat, keys):
        ax.plot(frame['step'], frame[key], label=arm)
        ax.set(title=key, xlabel='update')
        ax.grid(alpha=0.2)
axes.flat[0].legend()
fig.tight_layout()
plt.show()
